# EX — RAG (Retrieval-Augmented Generation) Real-World Exercise

Build a minimal, fully-offline RAG pipeline: chunk documents, embed (TF-IDF stand-in),
retrieve top-k, then construct a grounded prompt for an LLM (mocked). This mirrors the
real pipeline you'd build with a vector DB + real embedding model + real LLM API.


In [ ]:
import re, json, numpy as np
from collections import Counter

knowledge_base = [
    "Our return policy allows returns within 30 days of purchase with a valid receipt.",
    "Refunds are processed within 5-7 business days to the original payment method.",
    "Subscriptions can be cancelled anytime from Account Settings > Billing.",
    "The mobile app requires iOS 15+ or Android 10+ to install.",
    "For password resets, use the 'Forgot Password' link on the login page.",
    "Business accounts get a dedicated support line available 9am-6pm EST.",
    "Shipping typically takes 3-5 business days within the continental US.",
    "International orders may incur customs fees not included in the checkout price.",
]


## 1. Chunking
Our docs are already short, but real documents need splitting. **Pointer:** use overlap so answers near a boundary aren't lost.

In [ ]:
def chunk_text(text, max_words=40, overlap=10):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + max_words
        chunks.append(" ".join(words[start:end]))
        start += max_words - overlap
    return chunks

# For this KB, each entry is already a chunk. In a real system you'd chunk_text() each document first.
chunks = knowledge_base


## 2. Embed & Index (TF-IDF stand-in for a real embedding model)

In [ ]:
def tokenize(t): return re.findall(r"[a-z]+", t.lower())

vocab = sorted(set(w for c in chunks for w in tokenize(c)))
vidx = {w:i for i,w in enumerate(vocab)}
doc_freq = Counter()
for c in chunks:
    for w in set(tokenize(c)):
        doc_freq[w]+=1

def embed(text):
    vec = np.zeros(len(vocab))
    tf = Counter(tokenize(text))
    for w,count in tf.items():
        if w in vidx:
            idf = np.log((1+len(chunks))/(1+doc_freq[w]))+1
            vec[vidx[w]] = count*idf
    return vec

chunk_vectors = np.array([embed(c) for c in chunks])


### TODO 1
Write `retrieve(query, k=3)` returning the top-k chunks by cosine similarity (reuse the `cosine_sim` pattern from the Embeddings section).

In [ ]:
def cosine_sim(a,b):
    denom = np.linalg.norm(a)*np.linalg.norm(b)
    return 0.0 if denom==0 else np.dot(a,b)/denom

# TODO
def retrieve(query, k=3):
    pass

print(retrieve("how do I get my money back"))


<details><summary>Solution</summary>

```python
def retrieve(query, k=3):
    qvec = embed(query)
    sims = [cosine_sim(qvec, cv) for cv in chunk_vectors]
    ranked = sorted(zip(chunks, sims), key=lambda x: -x[1])
    return [c for c, s in ranked[:k]]
```
</details>


## 3. Grounded Prompt Construction
**Pointer:** always separate retrieved context from the question, and instruct the model to decline if the answer isn't present.

In [ ]:
class MockLLM:
    def complete(self, prompt):
        if "no relevant information" in prompt.lower():
            pass
        # naive mock: just echo which context sentence seems most relevant
        return "[mock answer grounded in provided context]"

llm = MockLLM()

def build_rag_prompt(query, context_chunks):
    context = "\n".join(f"- {c}" for c in context_chunks)
    return f'''Answer the question using ONLY the context below. If the answer is not in the context, say "I don't know based on the available information."

Context:
{context}

Question: {query}
Answer:'''

def rag_answer(query, k=3):
    context_chunks = retrieve(query, k=k)
    prompt = build_rag_prompt(query, context_chunks)
    return llm.complete(prompt), context_chunks

answer, used_chunks = rag_answer("What are your shipping times?")
print(answer)
print(used_chunks)


### TODO 2
Test the pipeline with a question that has **no answer** in the knowledge base (e.g., 'do you offer gift wrapping?'). Confirm the retrieved chunks are all low-relevance, and think about how you'd detect this automatically (hint: a similarity threshold).

In [ ]:
# TODO: call rag_answer() with an out-of-scope question and print the similarity scores for retrieved chunks


<details><summary>Solution / discussion</summary>

```python
query = "do you offer gift wrapping?"
qvec = embed(query)
sims = sorted([cosine_sim(qvec, cv) for cv in chunk_vectors], reverse=True)
print(sims[:3])
```
If the top similarity scores are all low (e.g. < 0.1), that's a signal to skip generation
entirely and return "I don't know" rather than risking a hallucinated answer — a simple,
effective guardrail.
</details>


## Key Takeaways
- RAG = retrieve relevant chunks, then force the LLM to answer using only that context.
- Always keep retrieved context and the question visually separated in the prompt.
- A similarity threshold before generation is a cheap, effective hallucination guardrail.
- Log the retrieved chunks alongside every answer — it's your primary debugging tool for bad answers.
